# Entanglement distillation -- 2 Node

Note: Entanglement distillation is enabled by default because the rules for distillation is generated and loaded by default. See [code](https://github.com/sequence-toolbox/SeQUeNCe/blob/ffd7c837f932c7bdc9450cd211aaf75b4d6a99a5/sequence/resource_management/resource_manager.py#L193)

In order to trigger the execution of entanglement distillation, the `target_fidelity` in the `app.start()` needs to be larger than the `raw_fidelity` of the newly generated entanglement pairs, commonly set in the json files.

``raw_fidelity`` refers to the initial fidelity of newly generated entanglement pairs between two neighbor nodes.

``targat_fidelity`` refers to the fidelity that the request wants.

**The imports**

In [45]:
from request_app import RequestApp

import numpy as np
import sequence.utils.log as log
from sequence.topology.router_net_topo import RouterNetTopo
from sequence.constants import SINGLE_HERALDED, SECOND, BELL_DIAGONAL_STATE_FORMALISM
from sequence.entanglement_management.generation import EntanglementGenerationA, EntanglementGenerationB
from sequence.entanglement_management.purification import BBPSSWProtocol

EntanglementGenerationA.set_global_type(SINGLE_HERALDED)
EntanglementGenerationB.set_global_type(SINGLE_HERALDED)
BBPSSWProtocol.set_formalism(BELL_DIAGONAL_STATE_FORMALISM)

**The Main**

In [46]:
def main(target_fidelity: float = 0.8):

    network_topo = RouterNetTopo(config_source="distillation_2node.json")
    tl = network_topo.get_timeline()

    # set up log
    log_filename = "distillation_2node.log"
    log.set_logger(__name__, tl, log_filename)
    log.set_logger_level('DEBUG')
    modules = ['bbpssw_bds']
    for module in modules:
        log.track_module(module)

    name_to_app = {}
    for router in network_topo.get_nodes_by_type(RouterNetTopo.QUANTUM_ROUTER):
        name_to_app[router.name] = RequestApp(router)
    
    tl.init()
    alice = "router_0"
    bob = "router_1"
    name_to_app[alice].start(responder=bob, start_t=1 * SECOND, end_t=2 * SECOND, memo_size=2, fidelity=target_fidelity)
    tl.run()

    raw_fidelity = 0.79  # see the distilation_2node.log
    print(f"Raw fidelity is: {raw_fidelity}")
    print(f"Target fidelity is: {target_fidelity}")

    print(f"Entangled pair count between Alice and Bob: {name_to_app[alice].memory_counter}")
    fidelities = np.array(name_to_app[alice].fidelities)
    if fidelities.size == 0:
        print("No accepted entangled pairs; fidelity statistics are unavailable.")
    else:
        average_fidelity = fidelities.mean()
        standard_deviation = fidelities.std()
        print(f"Average fidelity of the entangled pairs is: {average_fidelity:.6f}")
        print(f"Standard deviation of the fidelities is:    {standard_deviation:.6f}")

#### Case 1: target_fidelity < raw_fidelity

**Dstillation not activated**, generates 109 entanglement pairs within 1 second duration

In [47]:
main(target_fidelity=0.7)

Raw fidelity is: 0.79
Target fidelity is: 0.7
Entangled pair count between Alice and Bob: 109
Average fidelity of the entangled pairs is: 0.790000
Standard deviation of the fidelities is:    0.000000


#### Case 2: target_fidelity slightly larger than raw_fidelity


**Distillation is activated**, **achieves the target_fidelity**. 

But only 20 entanglement pairs are generated within 1 second duration, which is less than 109. Because purification improve fidelity at the cost of consuming entanglement pairs.

In [48]:
main(target_fidelity=0.8)

Raw fidelity is: 0.79
Target fidelity is: 0.8
Entangled pair count between Alice and Bob: 20
Average fidelity of the entangled pairs is: 0.819161
Standard deviation of the fidelities is:    0.005485


#### Case 3: target_fidelity is a lot larger than raw_fidelity


Distillation is activated, but **can never achieve the target_fidelity**. So 0 entanglement pairs are generated :(

In [49]:
main(target_fidelity=0.9)

Raw fidelity is: 0.79
Target fidelity is: 0.9
Entangled pair count between Alice and Bob: 0
No accepted entangled pairs; fidelity statistics are unavailable.
